# 01 - EDA\n
\n
Exploracao inicial do dataset HVFHV usando Spark SQL. Todas as transformacoes analiticas ficam em `spark.sql()`; pandas aparece apenas depois de agregacoes pequenas para visualizacao.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/notebooks')

import matplotlib.pyplot as plt
import seaborn as sns

from _lib import build_spark, SEED, DATA_GLOB, LOOKUP_PATH

spark = build_spark('nyc-rideshare-eda')
sns.set_theme(style='whitegrid')

In [ ]:
trips_df = spark.read.parquet(DATA_GLOB)\n
zones_df = (spark.read\n
    .option('header', True)\n
    .option('inferSchema', True)\n
    .csv(LOOKUP_PATH))\n
\n
trips_df.createOrReplaceTempView('trips_bronze')\n
zones_df.createOrReplaceTempView('taxi_zone_lookup')\n
\n
spark.sql('SELECT COUNT(*) AS total_rows FROM trips_bronze').show()

In [ ]:
spark.sql("""\n
SELECT\n
    MIN(request_datetime) AS min_request_datetime,\n
    MAX(dropoff_datetime) AS max_dropoff_datetime,\n
    COUNT(DISTINCT DATE(pickup_datetime)) AS active_days,\n
    COUNT(*) AS total_rows\n
FROM trips_bronze\n
""").show(truncate=False)\n
\n
spark.sql("""\n
SELECT\n
    DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS pickup_month,\n
    hvfhs_license_num,\n
    COUNT(*) AS trips\n
FROM trips_bronze\n
GROUP BY 1, 2\n
ORDER BY 1, 2\n
""").show(100, truncate=False)

In [ ]:
spark.sql("""\n
SELECT 'on_scene_datetime' AS column_name, SUM(CASE WHEN on_scene_datetime IS NULL THEN 1 ELSE 0 END) AS null_rows, COUNT(*) AS total_rows FROM trips_bronze\n
UNION ALL\n
SELECT 'originating_base_num', SUM(CASE WHEN originating_base_num IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\n
UNION ALL\n
SELECT 'tips', SUM(CASE WHEN tips IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\n
UNION ALL\n
SELECT 'airport_fee', SUM(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\n
UNION ALL\n
SELECT 'driver_pay', SUM(CASE WHEN driver_pay IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\n
""").show(truncate=False)

In [ ]:
spark.sql("""\n
SELECT\n
    PERCENTILE_APPROX(trip_miles, ARRAY(0.01, 0.5, 0.99)) AS trip_miles_pct,\n
    PERCENTILE_APPROX(trip_time, ARRAY(0.01, 0.5, 0.99)) AS trip_time_pct,\n
    PERCENTILE_APPROX(base_passenger_fare, ARRAY(0.01, 0.5, 0.99)) AS fare_pct,\n
    PERCENTILE_APPROX(driver_pay, ARRAY(0.01, 0.5, 0.99)) AS driver_pay_pct\n
FROM trips_bronze\n
WHERE trip_miles IS NOT NULL\n
  AND trip_time IS NOT NULL\n
  AND base_passenger_fare IS NOT NULL\n
""").show(truncate=False)\n
\n
spark.sql("""\n
SELECT\n
    COUNT(*) AS invalid_temporal_rows\n
FROM trips_bronze\n
WHERE pickup_datetime < request_datetime\n
   OR dropoff_datetime <= pickup_datetime\n
""").show()

In [ ]:
hourly_pdf = spark.sql("""\n
SELECT\n
    HOUR(pickup_datetime) AS pickup_hour,\n
    COUNT(*) AS trips\n
FROM trips_bronze\n
GROUP BY 1\n
ORDER BY 1\n
""").toPandas()\n
\n
ax = sns.lineplot(data=hourly_pdf, x='pickup_hour', y='trips', marker='o')\n
ax.set(title='Volume de corridas por hora', xlabel='Hora do dia', ylabel='Corridas')\n
plt.show()

In [ ]:
top_pickups_pdf = spark.sql("""\n
SELECT\n
    z.Zone,\n
    z.Borough,\n
    COUNT(*) AS trips\n
FROM trips_bronze t\n
LEFT JOIN taxi_zone_lookup z\n
  ON t.PULocationID = z.LocationID\n
WHERE t.PULocationID NOT IN (264, 265)\n
GROUP BY 1, 2\n
ORDER BY trips DESC\n
LIMIT 10\n
""").toPandas()\n
\n
ax = sns.barplot(data=top_pickups_pdf, x='trips', y='Zone', hue='Borough', dodge=False)\n
ax.set(title='Top 10 zonas de pickup', xlabel='Corridas', ylabel='Zona')\n
plt.show()

In [ ]:
borough_od_pdf = spark.sql("""\n
SELECT\n
    pu.Borough AS pu_borough,\n
    do.Borough AS do_borough,\n
    COUNT(*) AS trips\n
FROM trips_bronze t\n
LEFT JOIN taxi_zone_lookup pu\n
  ON t.PULocationID = pu.LocationID\n
LEFT JOIN taxi_zone_lookup do\n
  ON t.DOLocationID = do.LocationID\n
WHERE t.PULocationID NOT IN (264, 265)\n
  AND t.DOLocationID NOT IN (264, 265)\n
GROUP BY 1, 2\n
""").toPandas()\n
\n
pivot = borough_od_pdf.pivot(index='pu_borough', columns='do_borough', values='trips').fillna(0)\n
plt.figure(figsize=(10, 6))\n
sns.heatmap(pivot, cmap='Blues', fmt='.0f')\n
plt.title('Matriz origem-destino entre boroughs')\n
plt.show()

In [ ]:
spark.sql("""\n
SELECT\n
    corr(trip_miles, base_passenger_fare) AS corr_miles_fare,\n
    corr(trip_time, base_passenger_fare) AS corr_time_fare,\n
    corr(driver_pay, base_passenger_fare) AS corr_driver_pay_fare,\n
    corr(tips, base_passenger_fare) AS corr_tips_fare\n
FROM trips_bronze\n
WHERE trip_miles IS NOT NULL\n
  AND trip_time IS NOT NULL\n
  AND base_passenger_fare IS NOT NULL\n
""").show(truncate=False)

## Observacoes para o log de decisoes\n
\n
- Quantificar nulos de `on_scene_datetime` por operadora antes de descartar esse sinal de modelagem.\n
- Documentar a participacao dominante da Uber para evitar comparacoes brutas entre operadoras.\n
- Registrar o percentual removido por inconsistencias temporais e por velocidades extremas no notebook de preprocessing.

In [ ]:
spark.stop()